# Team 14: 512-token HotpotQA retrieval baseline

Colab-first notebook: HotpotQA distractor contexts → 512-token chunks → BGE-M3 embeddings → FAISS → question top-5 chunks. Generated artifacts are written to Google Drive.

## 1. Colab setup and persistent storage

Set `GITHUB_REPO_URL` before running if the notebook is not already inside a cloned checkout. The URL is intentionally a placeholder because the team's repository URL was not provided.

In [ ]:
from pathlib import Path
import os
import subprocess

GITHUB_REPO_URL = ""  # Example: https://github.com/<org>/<repo>.git
REPO_NAME = "adaptive-rag"
COLAB_ROOT = Path('/content')
repo_dir = COLAB_ROOT / REPO_NAME

if GITHUB_REPO_URL and not (repo_dir / 'src').exists():
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, str(repo_dir)], check=True)
elif not (repo_dir / 'src').exists():
    # Supports notebooks opened from an existing checkout or uploaded with the repo.
    candidates = [Path.cwd(), Path.cwd().parent]
    repo_dir = next((p for p in candidates if (p / 'src').exists()), repo_dir)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    artifact_root = Path('/content/drive/MyDrive/team14_adaptive_rag')
except ImportError:
    artifact_root = repo_dir / 'colab_artifacts'

artifact_root.mkdir(parents=True, exist_ok=True)
print('Repository:', repo_dir)
print('Persistent artifacts:', artifact_root)

In [ ]:
# Install from the checked-out repository. This downloads packages in Colab only.
subprocess.run(['pip', 'install', '-q', '-r', str(repo_dir / 'requirements.txt')], check=True)
import sys
sys.path.insert(0, str(repo_dir))

from src.config import artifact_paths, select_device
paths = artifact_paths(artifact_root)
os.environ['HF_HOME'] = str(paths['hf_cache'])
os.environ['TRANSFORMERS_CACHE'] = str(paths['hf_cache'])
print(paths)

In [ ]:
PREFERRED_DEVICE = 'auto'  # Change to 'cpu' or 'cuda' when needed.
DEVICE = select_device(PREFERRED_DEVICE)
print('Selected device:', DEVICE)
try:
    import torch
    if DEVICE == 'cuda':
        print(torch.cuda.get_device_name(0))
except ImportError:
    pass

## 2. Load the 50-question development sample

This downloads HotpotQA into the Colab/Hugging Face cache. The corpus is pooled and deduplicated from the selected questions' distractor contexts.

In [ ]:
from src.data import load_hotpotqa
from src.chunking import chunk_documents
from src.embeddings import BGEEmbedder
from src.retrieval import FaissRetriever

SAMPLE_SIZE = 50
SEED = 14
CHUNK_SIZE = 512
TOP_K = 5
examples, documents = load_hotpotqa(sample_size=SAMPLE_SIZE, seed=SEED)
print(f'{len(examples)} questions, {len(documents)} pooled documents')

In [ ]:
# BGE-M3 is downloaded into Google Drive cache, not the Mac.
embedder = BGEEmbedder(model_name='BAAI/bge-m3', device=DEVICE)
chunks = chunk_documents(documents, embedder.tokenizer, chunk_size=CHUNK_SIZE)
embeddings = embedder.embed_chunks(chunks)
retriever = FaissRetriever.build(chunks, embeddings)
index_dir = paths['indexes'] / 'hotpotqa_512'
retriever.save(index_dir)
print(f'{len(chunks)} chunks indexed; embedding dimension={embeddings.shape[1]}')
print('FAISS index:', index_dir / 'index.faiss')
print('Metadata:', index_dir / 'metadata.json')

## 3. Question → Top-5 retrieved chunks

Illustrative format only (not experiment results): an input question followed by ranked Wikipedia evidence chunks. This milestone does not generate an answer.

In [ ]:
import json
import time
from src.evaluation import retrieved_token_count

question = examples[0].question
started = time.perf_counter()
results = retriever.search(question, embedder, top_k=TOP_K)
latency_ms = (time.perf_counter() - started) * 1000
print('Input question')
print(question)
print('\nRetrieved evidence')
print(f'retrieval latency: {latency_ms:.2f} ms; retrieved tokens: {retrieved_token_count([r["chunk"] for r in results])}')
for rank, result in enumerate(results, 1):
    chunk = result['chunk']
    print(f'\nChunk {rank} — {chunk.title} | Rank {rank}')
    print(f'score={result["score"]:.4f}; chunk={chunk.chunk_id}; tokens={chunk.token_count}')
    print(chunk.text[:1000])

output = {
    'question': question,
    'top_k': TOP_K,
    'latency_ms': latency_ms,
    'retrieved_token_count': retrieved_token_count([r['chunk'] for r in results]),
    'results': [
        {'score': r['score'], 'chunk': r['chunk'].__dict__} for r in results
    ],
}
output_path = paths['results'] / 'baseline_512_first_question.json'
output_path.write_text(json.dumps(output, indent=2), encoding='utf-8')
print('Saved retrieval output:', output_path)